In [1]:
# ResourceMonitor for Jupyter
# Tracks: process CPU %, process RAM (MB), system CPU %, system RAM %
# Usage:
#   from time import sleep
#   mon = ResourceMonitor(interval=1.0).start()
#   # ... run cells / code ...
#   sleep(10)
#   mon.stop()
#   df = mon.to_dataframe(); df.tail()
#   mon.to_csv("usage_log.csv")

import os
import time
import threading
from datetime import datetime

try:
    import psutil
except ModuleNotFoundError:
    raise SystemExit(
        "psutil is required. Install with:\n  %pip install psutil"
    )

import pandas as pd


class ResourceMonitor:
    def __init__(self, interval: float = 1.0, include_children: bool = False):
        """
        interval: seconds between samples
        include_children: if True, include child processes of the kernel (e.g., subprocs) in process stats
        """
        self.interval = float(interval)
        self.include_children = include_children
        self._pid = os.getpid()
        self._proc = psutil.Process(self._pid)
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.Lock()
        self._rows = []  # list of dicts

    def _collect_once(self):
        # Ensure CPU% deltas are meaningful: warm up one call
        self._proc.cpu_percent(None)
        psutil.cpu_percent(None)

    def _gather(self):
        # Optionally sum children resource usage
        proc_cpu = self._proc.cpu_percent(None)
        proc_mem_bytes = self._proc.memory_info().rss
        if self.include_children:
            for ch in self._proc.children(recursive=True):
                try:
                    proc_cpu += ch.cpu_percent(None)
                    proc_mem_bytes += ch.memory_info().rss
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    continue

        sys_cpu = psutil.cpu_percent(None)
        vm = psutil.virtual_memory()
        now = datetime.now().isoformat(timespec="seconds")
        return {
            "timestamp": now,
            "proc_cpu_pct": proc_cpu,
            "proc_mem_mb": proc_mem_bytes / (1024 * 1024),
            "sys_cpu_pct": sys_cpu,
            "sys_mem_pct": vm.percent,
        }

    def _run(self):
        # Prime CPU counters
        self._collect_once()
        next_t = time.perf_counter()
        while not self._stop.is_set():
            try:
                row = self._gather()
                with self._lock:
                    self._rows.append(row)
            except psutil.Error:
                # If the process disappears or psutil hiccups, keep going
                pass
            next_t += self.interval
            # Sleep until the next tick (avoids drift)
            delay = max(0.0, next_t - time.perf_counter())
            self._stop.wait(delay)

    def start(self):
        if self._thread and self._thread.is_alive():
            return self
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, name="ResourceMonitor", daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=self.interval * 2)
        return self

    # Context manager sugar: with ResourceMonitor(...) as mon: ...
    def __enter__(self):
        return self.start()

    def __exit__(self, exc_type, exc, tb):
        self.stop()

    def to_dataframe(self) -> pd.DataFrame:
        with self._lock:
            return pd.DataFrame(self._rows).copy()

    def to_csv(self, path: str):
        df = self.to_dataframe()
        df.to_csv(path, index=False)

    def clear(self):
        with self._lock:
            self._rows.clear()

    def last(self):
        with self._lock:
            return self._rows[-1] if self._rows else None


In [2]:
SEED_DATA_SIZE = 5

In [3]:
from time import sleep
mon = ResourceMonitor(interval=1.0).start()

# Enhanced Summary Knowledge Tuning - Data Generation

## Overview

This notebook demonstrates how to generate high-quality knowledge tuning datasets using the SDG Hub framework. It creates multiple types of document augmentations and corresponding question-answer pairs that can be used to train or fine-tune language models for enhanced summarization and knowledge extraction capabilities.

## What This Notebook Does

This notebook will:

2. **Generate Four Types of Knowledge Tuning Datasets**:
   - **Extractive Summaries**: Concise summaries that extract key information directly from source documents
   - **Detailed Summaries**: Comprehensive summaries that provide thorough coverage of document content
   - **Key Facts**: Structured fact extraction with corresponding Q&A pairs
   - **Document-Based Q&A**: Question-answer pairs generated directly from document content


4. **Output Structured Training Data**:
   - For each augmentation we save JSONL dataset.
   - You can follow [knowledge_mixing](knowledge_mixing.ipynb) to convert it into training dataset

## Prerequisites

- SDG Hub installed and configured
- Environment variables set up (see [.env.example](.env.example)). Specifically set the model provider, seed data and output path.
- Document pre-processing completed (run [document_pre_processing.ipynb](document_pre_processing.ipynb) first)

```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
copy the .env.example to .env and set the model endpoint and generation/mixing parameters
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

## Next Steps

After running this notebook, use [knowledge_mixing](knowledge_mixing.ipynb) to combine and curate the generated datasets for final model training.


In [4]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
import os

# Load environment variables from .env file
load_dotenv()

/home/lab/rawhad/sdg_hub/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [5]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()  

In [6]:
def create_seed_data_from_quality_benchmark(run_on_validation=None, seed_data_path=None):
    """
    Create seed data from QuALITY Benchmark dataset.
    
    Args:
        run_on_validation (bool, optional): If True, use validation subset. If None, reads from env.
        seed_data_path (str, optional): Path to save seed data. If None, reads from env.
    
    Returns:
        datasets.Dataset: The processed corpus
    """
    # Use environment variables as defaults if not provided
    if run_on_validation is None:
        run_on_validation = os.getenv('RUN_ON_VALIDATION_SET', 'true').lower() == 'true'
    if seed_data_path is None:
        seed_data_path = os.getenv('SEED_DATA_PATH', 'seed_data_val.jsonl')
    
    # Load QuALITY Benchmark dataset
    print("Loading QuALITY Benchmark dataset...")
    quality_corpus = load_dataset("zitongyang/entigraph-quality-corpus", split='train').remove_columns(['entity', 'entigraph']).rename_columns({'raw': 'document', 'uid': 'document_outline'})
    
    # Define seed examples for knowledge tuning
    seed_examples = {
        "icl_document": (
          "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting a decline in tourism and fishing industries. Residents have organized weekly clean-up initiatives, but the scale of the problem overwhelms their efforts.",
          "Technologists at the local university have developed an AI-powered buoy system to combat this. The buoys, equipped with solar panels and filtration technology, can identify and absorb oil spills while collecting microplastics. Data from the buoys is shared publicly, raising awareness and pressuring corporations to adopt sustainable practices. Though costly, the project has sparked hope for revitalizing the ecosystem and economy."
        ),
        "icl_query_1": "How does the technological solution address the economic *and* environmental challenges highlighted in the document?",
        "icl_query_2": "What implicit values or priorities do the community's actions (clean-up initiatives) and the technologists' project reflect, and how do these align or contrast?",
        "icl_query_3": "Imagine the buoy project succeeds. What unintended consequences might arise from its impact, considering document's themes?",
        "domain": "articles/essays"
    }
    
    # Add seed examples to the corpus
    quality_corpus = quality_corpus.map(lambda x: seed_examples)
    
    if run_on_validation:
        # Validation set - use predefined document IDs for consistent evaluation
        DOC_UIDS = [
            ' Defining Decay Down by David Plotz',
            ' Fight Clubbed by David Plotz',
            ' I, Antichrist? by Jeffrey Goldberg',
            " It's Time To Keelhaul U-Haul! by Jeffrey Goldberg",
            " My Father's Estate by Ben Stein",
            '"Phone Me in Central Park" by McConnell, James V.',
            'A Coffin for Jacob by Ludwig, Edward W.',
            'A Fall of Glass by Lee, Stanley R.',
            'A Filbert Is a Nut by Raphael, Rick',
            'A Gift from Earth by Banister, Manly',
            'A Gleeb for Earth by Schafhauser, Charles',
            'A Good Year for the Roses? by David Edelstein',
            'A Pail of Air by Leiber, Fritz',
            'A Planet Named Joe by Hunter, Evan',
            "AI: what's the worst that could happen? by Harry Armstrong",
            'Accidental Death by Baily, Peter',
            'All Day September by Kuykendall, Roger',
            'Ambition by Bade, William L.',
            'And Then the Town Took Off by Wilson, Richard',
            'Atom Mystery [Young Atom Detective] by Coombs, Charles Ira',
            'Beach Scene by King, Marshall',
            'Big Ancestor by Wallace, F. L. (Floyd L.)',
            'Birds of a Feather by Silverberg, Robert',
            'Bodyguard by Gold, H. L. (Horace Leonard)'
        ]
        
        # Filter corpus to validation set
        quality_corpus = quality_corpus.filter(lambda x: x['document_outline'] in DOC_UIDS)
        print(f"Running on validation set with {len(quality_corpus)} documents")
    else:
        # Use full dataset for training
        print(f"Running on full dataset with {len(quality_corpus)} documents")
    
    # Save the seed data
    quality_corpus.to_json(seed_data_path, orient='records', lines=True)
    print(f"Saved seed data to: {seed_data_path}")
    
    return quality_corpus

In [7]:
# Create seed data using the function
quality_corpus = create_seed_data().select(range(SEED_DATA_SIZE))

Loading QuALITY Benchmark dataset...
Running on validation set with 24 documents


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 109.36ba/s]

Saved seed data to: seed_data_val.jsonl


### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [8]:
# Setup model configuration in flow object
def set_model_config(flow_object):
    model_provider = os.getenv('MODEL_PROVIDER', 'hosted_vllm')
    print(f"Using model provider: {model_provider}")
    # Set model provider
    if model_provider == 'hosted_vllm':    
        vllm_model = os.getenv('VLLM_MODEL', 'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct')
        vllm_model = 'hosted_vllm/llama33-70b'
        vllm_api_base = os.getenv('VLLM_API_BASE', 'http://localhost:8000/v1')
        vllm_api_base = 'http://localhost:30310/v1'
        vllm_api_key = os.getenv('VLLM_API_KEY', 'EMPTY')
        enable_reasoning = os.getenv('ENABLE_REASONING', 'false').lower() in ('1', 'true', 'yes')
        print(f"Using reasoning: {enable_reasoning}")
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
            enable_reasoning=enable_reasoning,
        )
    elif model_provider == 'openai':
        openai_api_key = os.getenv('OPENAI_API_KEY')
        openai_model = os.getenv('OPENAI_MODEL', 'openai/gpt-4')
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
        )
    elif model_provider == 'ollama':
        ollama_model = os.getenv('OLLAMA_MODEL', 'ollama/gemma2')
        ollama_api_base = os.getenv('OLLAMA_API_BASE', 'http://localhost:11434')
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
        )
    elif model_provider == 'maas':
        maas_model = os.getenv('MAAS_MODEL')
        maas_api_base = os.getenv('MAAS_API_BASE')
        maas_api_key = os.getenv('MAAS_API_KEY')
        flow_object.set_model_config(
            model=maas_model,
            api_base=maas_api_base,
            api_key=maas_api_key,
        )
    return flow_object 

#### Discover the available generation flows

In [9]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[21:06:28] INFO     Discovered 7 flows                                                              ]8;id=898424;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=582996;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/registry.py#113\113]8;;\

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID               ┃ Name                  ┃ Author               ┃ Tags                  ┃ Description           ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397 │ Advanced Japanese     │ SDG Hub Contributors │ question-generation,  │ A comprehensive flow  │
│                  │ Document Grounded     │                      │ knowledge-extraction, │ that generates        │
│                  │ Question-Answer       │                      │ qa-pairs,             │ high-quality          │
│                  │ Generation Flow for   │                      │ document-processing,  │ question-answer pairs │
│                  │ Knowledge Tuning      │                      │ educational, japanese │ from Japanese input   │
│                  │                       │                      │                       │ documents using       │
│                  │                       │                      │                       │ multiple LLM blocks   │
│                  │                       │                      │                       │ for question          │
│                  │                       │                      │                       │ generation, answer    │
│                  │                       │                      │                       │ synthesis, and        │
│                  │                       │                      │                       │ quality evaluation.   │
│ epic-jade-656    │ Extractive Summary    │ SDG Hub Contributors │ knowledge-tuning,     │ Generate extractive   │
│                  │ Knowledge Tuning      │                      │ document-internaliza… │ summary from the      │
│                  │ Dataset Generation    │                      │ question-generation,  │ input document. Each  │
│                  │ Flow                  │                      │ knowledge-extractive… │ document is first     │
│                  │                       │                      │ qa-pairs,             │ converted into list   │
│                  │                       │                      │ extractive-summaries  │ of knowledge segments │
│                  │                       │                      │                       │ for creating          │
│                  │                       │                      │                       │ extractive summary    │
│                  │                       │                      │                       │ and then annotated    │
│                  │                       │                      │                       │ with context,         │
│                  │                       │                      │                       │ relationship and      │
│                  │                       │                      │                       │ relevance. This is    │
│                  │                       │                      │                       │ then converted into   │
│                  │                       │                      │                       │ Question-Answer       │
│                  │                       │                      │                       │ pairs.                │
│ green-clay-812   │ Structured Text       │ SDG Hub Contributors │ text-analysis,        │ Multi-step pipeline   │
│                  │ Insights Extraction   │                      │ summarization, nlp,   │ for extracting        │
│                  │ Flow                  │                      │ structured-output,    │ structured insights   │
│                  │                       │                      │ insights,             │ from text including   │
│                  │                       │                      │ sentiment-analysis,   │ summary, keywords,    │
│                  │                       │            

Available flows: [{'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'clean-shadow-397', 'name': 'Advanced Japanese Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'id': 'green-clay-812', 'name': 'Structured Text Insights Extraction Flow'}]
QA flows: [{'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'clean-shadow-397', 'name': 'Advanced Japanese Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'epic-jade-656', 'na

In [10]:
# We will use below mapping of flow names to their respective summarization flows
flow_name_map = {
        'Detailed Summary Knowledge Tuning Dataset Generation Flow': 'gen_detailed_summary',
        'Extractive Summary Knowledge Tuning Dataset Generation Flow': 'gen_extractive_summary',
    }

In [11]:
# Get runtime parameters
enable_reasoning = os.getenv('ENABLE_REASONING', 'false').lower() in ('1', 'true', 'yes')
number_of_summaries = int(os.getenv('NUMBER_OF_SUMMARIES', '50'))
max_concurrency = int(os.getenv('MAX_CONCURRENCY', '50'))
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')

In [ ]:
# Generate data for extractive summary
flow_name = "Extractive Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
number_of_summaries = int(os.getenv('NUMBER_OF_SUMMARIES', '50'))
# Generate data for extractive summary
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        'question_generation': {'max_tokens': 1024}, 
        'gen_extractive_summary': {'n': number_of_summaries, 'max_tokens': 6000}
        }
else:
    runtime_params = {
    'gen_extractive_summary': {
        'n': number_of_summaries
    }
}

extractive_summary_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency)

extractive_summary_generated_data.to_json(os.path.join(save_data_path, 'extractive_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")

print(f"✓ Columns: {list(extractive_summary_generated_data.column_names)}")

[21:06:28] INFO     Loading flow from:                                                                  ]8;id=282587;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=593985;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#140\140]8;;\
                    /workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/flows/qa_generation/document_grounde            
                    d_qa/enhanced_multi_summary_qa/extractive_summary/flow.yaml                                    

Using model provider: hosted_vllm
Using reasoning: False


           INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',                 ]8;id=390990;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=254582;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#821\821]8;;\
                    'eval_faithful_llm_chat', 'gen_extractive_summary', 'question_generation']                     

           INFO     Successfully configured 4 LLM blocks with: model: 'hosted_vllm/llama33-70b',        ]8;id=160206;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=256421;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#863\863]8;;\
                    api_base: 'http://localhost:30310/v1', api_key: EMPTY, enable_reasoning: False                 

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',                  ]8;id=846742;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=57732;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#866\866]8;;\
                    'gen_extractive_summary', 'question_generation']                                               

...

╭──────────────────────────────────────── eval_faithful_prompt - Complete ────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 1,154 → 1,154                                                                                             │
│ Columns: 20 → 21                                                                                                │
│ 🟢 Added: eval_faithful_prompt                                                                                  │
│ 📋 Final Columns: answer_generation_prompt, base_document, document, document_outline, domain,                  │
│ eval_faithful_prompt, extract_answers_content, extract_extractive_summary_content, extract_questions_content,   │
│ extractive_summary_prompt, icl_document, icl_query_1, icl_query_2, icl_query_3, question,                       │
│ question_generation_prompt, question_list, raw_document, raw_summary, response, response_dict                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:36:26] INFO     Block 'eval_faithful_prompt' completed successfully: 1154 samples, 21 columns       ]8;id=333007;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=386167;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#689\689]8;;\

           INFO     Executing block 16/19: eval_faithful_llm_chat (LLMChatBlock)                        ]8;id=360051;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=817220;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/flow/base.py#627\627]8;;\

╭──────────────────────────────────────────── eval_faithful_llm_chat ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: LLMChatBlock                                                                                        │
│ Input Rows: 1,154                                                                                               │
│ Input Columns: 21                                                                                               │
│ Column Names: document_outline, raw_document, icl_document, icl_query_1, icl_query_2, icl_query_3, domain,      │
│ base_document, extractive_summary_prompt, raw_summary, extract_extractive_summary_content, document,            │
│ question_generation_prompt, question_list, extract_questions_content, question, answer_generation_prompt,       │
│ response_dict, extract_answers_content, response, eval_faithful_prompt                                          │
│ Expected Output Columns: eval_faithful_response_dict                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[22:36:27] INFO     Starting async generation for 1154 samples (max_concurrency=2)            ]8;id=820716;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=647254;file:///workspace/home/lab/rawhad/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#208\208]8;;\

In [ ]:
# Generate similar data for Detailed Summary
flow_name = "Detailed Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        'question_generation': {'max_tokens': 1024}, 
        'gen_detailed_summary': {'n': number_of_summaries, 'max_tokens': 6000}
        }
else:
    runtime_params = ({'gen_detailed_summary': {
        'n': number_of_summaries
    }})
# Generate data for detailed summary
detailed_summary_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=50)

detailed_summary_generated_data.to_json(os.path.join(save_data_path, 'detailed_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")

print(f"✓ Columns: {list(detailed_summary_generated_data.column_names)}")

In [ ]:
# Generate similar data for key facts 
flow_name = "Key Facts Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
runtime_params = {}
if enable_reasoning:
    # Increase max tokens for Question Generation to accommodate reasoning content
    runtime_params = {
        'generate_key_fact_qa': {'max_tokens': 6000}, 
        }

# Generate data for key facts summary
key_facts_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency)

key_facts_generated_data.to_json(os.path.join(save_data_path, 'key_facts_to_qa', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.column_names)}")

In [ ]:
flow_name = "Document Based Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
runtime_params = {}
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        'question_generation': {'max_tokens': 2048}, 
        }

document_based_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency)
    
document_based_generated_data.to_json(os.path.join(save_data_path, 'document_based_qa', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Document based: {len(document_based_generated_data)} records")

print(f"✓ Columns: {list(document_based_generated_data.column_names)}")

🎉 You now have all three four of document augmentations (detailed summaries, extractive summaries, key facts and document based) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).

In [ ]:
sleep(10)
mon.stop()
df = mon.to_dataframe(); df.tail()
mon.to_csv(f"usage_log_{SEED_DATA_SIZE}.csv")